<!-- CELL 1 — MARKDOWN -->
# Phase 4 — Uplift Models
### Uplift Modeling on the Hillstrom Email Marketing Dataset

**Project:** `uplift-causal-ml` — a 7-phase causal machine learning project on uplift modeling.

**Recap:** Phase 1 loaded and validated the Hillstrom dataset; Phase 2 produced a stratified
80/20 train/test split; Phase 3 trained a naive baseline classifier predicting `visit`
directly (ignoring treatment entirely) -- AUC-ROC 0.6021, with a verified top-decile-vs-rest
gap of +10.36 pp.

**This notebook (Phase 4) covers three genuine uplift models**, each of which explicitly
estimates the *effect of treatment* on `visit`, rather than just predicting `visit` itself:

1. **Two-Model Approach** -- the simplest uplift method (two separate classifiers)
2. **Class Transformation Approach** -- the "revert label" trick (Gutierrez & Gerardy, 2017)
3. **Causal Forest** -- the principled, statistically-grounded main model
   (Athey, Tibshirani & Wager, 2019)

For each model, we run a signal verification check (top decile vs. bottom 90%, using
*real* treated-vs-control outcome gaps -- not predicted probabilities), then combine all
three uplift scores plus the Phase 3 baseline into one CSV for Phase 5, which will
formally evaluate all four with Qini coefficients.


<!-- CELL 2 — MARKDOWN -->
## 1. Load the Phase 2/3 split from `data/processed/`

In [1]:
# ---------------------------------------------------------------------------
# CELL 3 — CODE: Load X_train/X_test, treatment_train/treatment_test, and
# y_train/y_test DIRECTLY from data/processed/ -- the exact split Phases 2
# and 3 used. `visit` is the outcome throughout this notebook, consistent
# with Phase 3's corrected target (see Phase 3 notebook for why `conversion`
# was abandoned as a modeling target).
# ---------------------------------------------------------------------------
import os
import re
import subprocess
import sys

import joblib
import numpy as np
import pandas as pd
from lightgbm import LGBMClassifier, LGBMRegressor
from IPython.display import Markdown, display

processed_dir = os.path.join("..", "data", "processed")

X_train = pd.read_csv(os.path.join(processed_dir, "X_train.csv"))
X_test = pd.read_csv(os.path.join(processed_dir, "X_test.csv"))
treatment_train = pd.read_csv(os.path.join(processed_dir, "treatment_train.csv"))["treatment"]
treatment_test = pd.read_csv(os.path.join(processed_dir, "treatment_test.csv"))["treatment"]
y_train = pd.read_csv(os.path.join(processed_dir, "y_train.csv"))
y_test = pd.read_csv(os.path.join(processed_dir, "y_test.csv"))

outcome_col = "visit"
y_train_target = y_train[outcome_col]
y_test_target = y_test[outcome_col]

# Same column-name sanitization as Phase 3 -- LightGBM rejects special JSON
# characters (e.g. '$', ',', '(', ')') in feature names, which the Phase 2
# one-hot columns contain (e.g. "history_segment_7) $1,000 +").
def sanitize_columns(columns):
    return [re.sub(r"[^A-Za-z0-9_]+", "_", str(col)) for col in columns]

X_train.columns = sanitize_columns(X_train.columns)
X_test.columns = sanitize_columns(X_test.columns)

print(f"X_train: {X_train.shape}   X_test: {X_test.shape}")
print(f"Outcome column: '{outcome_col}'")
print(f"Train {outcome_col} rate: {y_train_target.mean()*100:.3f}%   Test {outcome_col} rate: {y_test_target.mean()*100:.3f}%")
print(f"Train treatment rate: {treatment_train.mean()*100:.2f}%   Test treatment rate: {treatment_test.mean()*100:.2f}%")


X_train: (51200, 18)   X_test: (12800, 18)
Outcome column: 'visit'
Train visit rate: 14.662%   Test visit rate: 14.742%
Train treatment rate: 66.71%   Test treatment rate: 66.71%


<!-- CELL 4 — MARKDOWN -->
## 2. Causal ML library setup

In [2]:
# ---------------------------------------------------------------------------
# CELL 5 — CODE: Import the causal forest library. We prefer `econml`
# (Microsoft's causal inference library, which implements CausalForestDML
# per Athey/Tibshirani/Wager 2019). If it isn't already installed, we try
# installing it at runtime; if that still fails (e.g. a locked-down or
# incompatible environment), we fall back to `causalml`'s
# UpliftRandomForestClassifier, a different (but conceptually related)
# tree-based uplift estimator. Whichever library ends up being used is
# printed clearly below, and noted again in the Phase 4 summary at the end.
# ---------------------------------------------------------------------------
CAUSAL_LIBRARY = None

try:
    from econml.dml import CausalForestDML
    CAUSAL_LIBRARY = "econml"
except ImportError:
    print("econml not found -- attempting to install it now (pip install econml)...")
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "econml", "-q"])
        from econml.dml import CausalForestDML
        CAUSAL_LIBRARY = "econml"
    except Exception as install_error:
        print(f"econml install/import failed ({install_error!r}). Falling back to causalml...")
        try:
            from causalml.inference.tree import UpliftRandomForestClassifier
            CAUSAL_LIBRARY = "causalml"
        except ImportError as causalml_error:
            raise ImportError(
                "Neither econml nor causalml could be installed/imported. Install one of "
                "them manually (`pip install econml` or `pip install causalml`) and re-run "
                "this notebook."
            ) from causalml_error

print(f"Causal forest library in use: {CAUSAL_LIBRARY}")
if CAUSAL_LIBRARY == "econml":
    print("(Preferred choice -- econml's CausalForestDML combines the Causal Forest / "
          "Generalized Random Forest algorithm with Double Machine Learning residualization.)")
else:
    print("(Fallback -- econml was unavailable in this environment; causalml's "
          "UpliftRandomForestClassifier is used instead. See Section 5 markdown for how "
          "its mechanics differ slightly from the econml path this notebook was designed "
          "and tested against.)")


Causal forest library in use: econml
(Preferred choice -- econml's CausalForestDML combines the Causal Forest / Generalized Random Forest algorithm with Double Machine Learning residualization.)


<!-- CELL 6 — MARKDOWN -->
## 3. Model 1 — Two-Model Approach

**What it is:** train two completely separate classifiers -- one on *only* the treated
training rows, one on *only* the control training rows -- both predicting `visit` from the
same features. For every test customer, ask both models "how likely is this customer to
visit?" and take the difference: `uplift = P(visit | treated model) - P(visit | control model)`.
This is the simplest possible uplift method, and a very common first attempt in practice.

**Known weakness:** each model only ever sees *half* the training data (the treated model
never sees a single control example and vice versa), so both are individually noisier than
a model trained on the full dataset would be. Worse, the final uplift score is the
*difference* of two independently noisy predictions -- small, unrelated errors in each
model don't cancel out, they **compound**, which is exactly why uplift estimates from this
approach tend to be noisy and why more sophisticated methods (Models 2 and 3 below) were
developed.


In [3]:
# ---------------------------------------------------------------------------
# CELL 7 — CODE: Two-Model Approach. Train separate classifiers on the
# treated-only and control-only training rows, predict both on the full
# test set, and take the difference as the uplift score.
# ---------------------------------------------------------------------------
train_treated_mask = treatment_train == 1
train_control_mask = treatment_train == 0

X_train_treated, y_train_treated = X_train[train_treated_mask], y_train_target[train_treated_mask]
X_train_control, y_train_control = X_train[train_control_mask], y_train_target[train_control_mask]

print(f"Treated-only training rows: {len(X_train_treated):,}  ({outcome_col} rate {y_train_treated.mean()*100:.2f}%)")
print(f"Control-only training rows: {len(X_train_control):,}  ({outcome_col} rate {y_train_control.mean()*100:.2f}%)")

# scale_pos_weight per submodel, same technique as Phase 3, since each
# submodel faces its own (still fairly imbalanced) positive class rate.
def make_scale_pos_weight(y):
    n_neg, n_pos = (y == 0).sum(), (y == 1).sum()
    return n_neg / n_pos

treated_model = LGBMClassifier(
    scale_pos_weight=make_scale_pos_weight(y_train_treated),
    random_state=42, n_estimators=200, verbosity=-1,
)
control_model = LGBMClassifier(
    scale_pos_weight=make_scale_pos_weight(y_train_control),
    random_state=42, n_estimators=200, verbosity=-1,
)

treated_model.fit(X_train_treated, y_train_treated)
control_model.fit(X_train_control, y_train_control)

treated_pred = treated_model.predict_proba(X_test)[:, 1]
control_pred = control_model.predict_proba(X_test)[:, 1]
uplift_two_model = treated_pred - control_pred

print(f"\nTwo-Model uplift score -- mean: {uplift_two_model.mean():+.4f}, "
      f"min: {uplift_two_model.min():+.4f}, max: {uplift_two_model.max():+.4f}")


Treated-only training rows: 34,155  (visit rate 16.73%)
Control-only training rows: 17,045  (visit rate 10.51%)

Two-Model uplift score -- mean: +0.0652, min: -0.5805, max: +0.7468


<!-- CELL 8 — MARKDOWN -->
## 4. Model 2 — Class Transformation Approach

**The "revert label" trick** (Gutierrez, P., & Gerardy, J-Y. (2017). *Causal Inference and
Uplift Modeling: A Review of the Literature*. Proceedings of Machine Learning Research.):
construct a single transformed target

$$Z = 1 \text{ if } (T=1 \text{ and } Y=1) \text{ or } (T=0 \text{ and } Y=0), \text{ else } Z = 0$$

i.e. `Z=1` exactly when a customer's treatment assignment "matched" a positive outcome, or
their control assignment "matched" a negative outcome. Train **one single classifier** on
all the training data to predict `Z`, and convert its predicted probability into an uplift
score with `uplift = 2 * P(Z=1) - 1` (this rescaling comes from the fact that, under
randomized treatment assignment with equal treat/control probabilities, `P(Z=1)` is related
to the true uplift by exactly this linear transform). Unlike the Two-Model Approach, this
uses **all** the training data in one model, which is its main advantage.


In [4]:
# ---------------------------------------------------------------------------
# CELL 9 — CODE: Class Transformation Approach. Build the revert-label
# target Z, train a single classifier on the FULL training set, and convert
# its predicted P(Z=1) into an uplift score via 2*P(Z=1) - 1.
# ---------------------------------------------------------------------------
Z_train = (
    ((treatment_train == 1) & (y_train_target == 1)) |
    ((treatment_train == 0) & (y_train_target == 0))
).astype(int)

print(f"Z=1 rate in training set: {Z_train.mean()*100:.2f}%  "
      f"({int(Z_train.sum()):,} / {len(Z_train):,})")
print("(Fairly balanced -- no special class-imbalance handling needed for this submodel, "
      "unlike the visit/conversion targets used elsewhere in this project.)")

class_transform_model = LGBMClassifier(random_state=42, n_estimators=200, verbosity=-1)
class_transform_model.fit(X_train, Z_train)

p_z1 = class_transform_model.predict_proba(X_test)[:, 1]
uplift_class_transform = 2 * p_z1 - 1

print(f"\nClass Transformation uplift score -- mean: {uplift_class_transform.mean():+.4f}, "
      f"min: {uplift_class_transform.min():+.4f}, max: {uplift_class_transform.max():+.4f}")


Z=1 rate in training set: 40.95%  (20,968 / 51,200)
(Fairly balanced -- no special class-imbalance handling needed for this submodel, unlike the visit/conversion targets used elsewhere in this project.)

Class Transformation uplift score -- mean: -0.1804, min: -0.7217, max: +0.5171


<!-- CELL 10 — MARKDOWN -->
## 5. Model 3 — Causal Forest (main model)

**Citation:** Athey, S., Tibshirani, J., & Wager, S. (2019). *Generalized Random Forests*.
The Annals of Statistics, 47(2), 1148-1178. This paper introduced the Generalized Random
Forest (GRF) framework, of which the Causal Forest is a special case. `econml`'s
`CausalForestDML` combines this with the **Double Machine Learning (DML)** framework for
additional robustness.

**In plain language, for explaining this in a viva:**

1. **Two "nuisance" models remove predictable noise first.** Before estimating any
   treatment effect, `CausalForestDML` trains two auxiliary models on the features `X`:
   - `model_y` predicts the outcome `Y` (visit) from `X` alone -- "how likely is this
     customer to visit, based on who they are, regardless of treatment?"
   - `model_t` predicts the treatment `T` from `X` alone -- "how likely is this customer to
     have been treated, based on who they are?" (In a properly randomized experiment like
     Hillstrom's, this should come out close to the constant assignment probability
     (~66.7%), since treatment shouldn't genuinely be predictable from `X`.)

   Both predictions are then subtracted off (**"residualized"**): what's left is the
   *outcome variation `Y` can't already be explained by `X`*, and the *treatment variation
   `T` can't already be explained by `X`*. This is the "Double Machine Learning" step -- it
   de-noises the problem before the causal step, and is what makes the final treatment-effect
   estimates statistically valid even if the nuisance models themselves aren't perfect.

2. **A forest is then grown to split on treatment-effect HETEROGENEITY, not on outcome.**
   A normal random forest tree splits data to make outcome predictions as accurate as
   possible within each leaf. A causal forest tree instead splits data to make the
   *residualized treatment effect* as different as possible *between* leaves -- each split
   is chosen specifically to separate "customers who respond a lot to the email" from
   "customers who respond little or not at all."

3. **"Honest" splitting avoids overfitting.** For each tree, one random subsample of the
   data decides *where* to split, and a separate subsample *estimates the treatment effect*
   within each resulting leaf. The effect estimate in a leaf is therefore never computed on
   the same data used to find that leaf's boundaries -- this is what gives the Causal Forest
   its (asymptotically) unbiased, statistically valid effect estimates, unlike the two
   heuristic models above.

4. **Output:** `.effect(X)` returns, for every customer, an estimate of their
   **individual/conditional average treatment effect (CATE)** -- literally "how much more
   likely is this specific customer to visit, because of the email, given their features."
   That is our uplift score for this model.


In [5]:
# ---------------------------------------------------------------------------
# CELL 11 — CODE: Fit the Causal Forest on the training set and get
# per-customer treatment effect estimates on the test set.
# ---------------------------------------------------------------------------
if CAUSAL_LIBRARY == "econml":
    # model_y: LightGBM regressor predicting visit (0/1) from X alone (nuisance model 1)
    # model_t: LightGBM classifier predicting treatment from X alone (nuisance model 2)
    # discrete_treatment=True tells econml that T is binary (0/1), so model_t must be a
    # classifier. n_estimators=200 must be divisible by subforest_size (default 4).
    causal_forest_model = CausalForestDML(
        model_y=LGBMRegressor(n_estimators=100, verbosity=-1, random_state=42),
        model_t=LGBMClassifier(n_estimators=100, verbosity=-1, random_state=42),
        discrete_treatment=True,
        n_estimators=200,
        random_state=42,
        cv=2,
        n_jobs=-1,
    )
    causal_forest_model.fit(y_train_target, treatment_train, X=X_train)
    uplift_causal_forest = causal_forest_model.effect(X_test)

else:  # CAUSAL_LIBRARY == "causalml" fallback
    # causalml's UpliftRandomForestClassifier expects string treatment labels and
    # fits/predicts directly on numpy arrays; its .predict() returns the estimated
    # uplift score per row directly (no separate nuisance-model step, since it isn't
    # a DML-based method -- see the note above on how this differs from econml's path).
    causal_forest_model = UpliftRandomForestClassifier(
        n_estimators=200, control_name="0", random_state=42,
    )
    causal_forest_model.fit(
        X_train.values,
        treatment_train.astype(str).values,
        y_train_target.values,
    )
    uplift_causal_forest = causal_forest_model.predict(X_test.values).flatten()

print(f"Causal Forest ({CAUSAL_LIBRARY}) uplift score -- mean: {uplift_causal_forest.mean():+.4f}, "
      f"min: {uplift_causal_forest.min():+.4f}, max: {uplift_causal_forest.max():+.4f}")


Causal Forest (econml) uplift score -- mean: +0.0622, min: -0.1903, max: +0.2780


<!-- CELL 12 — MARKDOWN -->
## 6. Signal verification for each model

Same idea as Phase 3's top-decile-vs-rest check, but adapted for uplift: instead of
comparing a predicted probability against actual outcomes directly, we compare each
model's **ranking** against the **real, observed uplift** (`treated visit rate - control
visit rate`, computed from actual test-set outcomes) within the top decile of that ranking
vs. the bottom 90%. A genuinely useful uplift model should rank customers such that the
top decile shows a *real* uplift gap meaningfully larger than the rest of the customers --
if it doesn't, the model isn't actually finding heterogeneous treatment effects, no matter
how sophisticated it is.

Note this check is inherently noisier than Phase 3's version: it requires enough treated
*and* control customers within each subgroup to compute a reliable rate difference, so we
report subgroup sizes alongside the results.


In [6]:
# ---------------------------------------------------------------------------
# CELL 13 — CODE: For each of the 3 uplift models, rank test customers by
# predicted uplift score, then compute the ACTUAL uplift (real treated vs.
# control visit rate) within the top decile vs. the bottom 90%. Flag if the
# gap isn't meaningfully larger than the base rate.
# ---------------------------------------------------------------------------
# Minimum gap (top-decile actual uplift minus bottom-90% actual uplift) to
# consider a model's ranking as carrying meaningful signal. Set lower than
# Phase 3's threshold (1pp vs. 2pp) because this is a noisier, second-order
# metric -- an uplift-of-a-subgroup, not a plain outcome rate.
MIN_MEANINGFUL_UPLIFT_GAP_PP = 0.01  # 1 percentage point

def actual_uplift(df, treat_col="actual_treatment", outcome_col_="actual_visit"):
    """Real (treated rate - control rate) within a subgroup of test rows."""
    treated = df.loc[df[treat_col] == 1, outcome_col_]
    control = df.loc[df[treat_col] == 0, outcome_col_]
    if len(treated) == 0 or len(control) == 0:
        return np.nan, len(treated), len(control)
    return treated.mean() - control.mean(), len(treated), len(control)

uplift_scores = {
    "Two-Model Approach": uplift_two_model,
    "Class Transformation": uplift_class_transform,
    "Causal Forest": uplift_causal_forest,
}

overall_actual_uplift, _, _ = actual_uplift(pd.DataFrame({
    "actual_treatment": treatment_test.values, "actual_visit": y_test_target.values,
}))
print(f"Overall actual uplift across the full test set: {overall_actual_uplift*100:+.2f} pp\n")

verification_results = {}
for model_name, scores in uplift_scores.items():
    ranked = pd.DataFrame({
        "actual_treatment": treatment_test.values,
        "actual_visit": y_test_target.values,
        "score": scores,
    }).sort_values("score", ascending=False).reset_index(drop=True)

    n = len(ranked)
    top_n = int(np.ceil(n * 0.10))
    top_decile = ranked.iloc[:top_n]
    bottom_90pct = ranked.iloc[top_n:]

    top_uplift, top_n_treat, top_n_control = actual_uplift(top_decile)
    bottom_uplift, bot_n_treat, bot_n_control = actual_uplift(bottom_90pct)
    gap = top_uplift - bottom_uplift
    passed = gap >= MIN_MEANINGFUL_UPLIFT_GAP_PP

    verification_results[model_name] = {
        "top_decile_uplift": top_uplift, "bottom_90pct_uplift": bottom_uplift,
        "gap": gap, "passed": passed,
    }

    print(f"--- {model_name} ---")
    print(f"  Top decile actual uplift    : {top_uplift*100:+.2f} pp  "
          f"(n_treated={top_n_treat:,}, n_control={top_n_control:,})")
    print(f"  Bottom 90% actual uplift    : {bottom_uplift*100:+.2f} pp  "
          f"(n_treated={bot_n_treat:,}, n_control={bot_n_control:,})")
    print(f"  Gap (top10 - bottom90)      : {gap*100:+.2f} pp")
    if passed:
        print(f"  SIGNAL CHECK PASSED (gap >= {MIN_MEANINGFUL_UPLIFT_GAP_PP*100:.0f} pp threshold)")
    else:
        print(f"  SIGNAL CHECK FAILED (gap < {MIN_MEANINGFUL_UPLIFT_GAP_PP*100:.0f} pp threshold) "
              "-- this model's ranking does not show meaningfully stronger uplift in its top decile.")
    print()


Overall actual uplift across the full test set: +5.56 pp

--- Two-Model Approach ---
  Top decile actual uplift    : +8.58 pp  (n_treated=868, n_control=412)
  Bottom 90% actual uplift    : +5.23 pp  (n_treated=7,671, n_control=3,849)
  Gap (top10 - bottom90)      : +3.36 pp
  SIGNAL CHECK PASSED (gap >= 1 pp threshold)

--- Class Transformation ---
  Top decile actual uplift    : +7.12 pp  (n_treated=872, n_control=408)
  Bottom 90% actual uplift    : +5.35 pp  (n_treated=7,667, n_control=3,853)
  Gap (top10 - bottom90)      : +1.77 pp
  SIGNAL CHECK PASSED (gap >= 1 pp threshold)

--- Causal Forest ---
  Top decile actual uplift    : +4.70 pp  (n_treated=851, n_control=429)
  Bottom 90% actual uplift    : +5.66 pp  (n_treated=7,688, n_control=3,832)
  Gap (top10 - bottom90)      : -0.96 pp
  SIGNAL CHECK FAILED (gap < 1 pp threshold) -- this model's ranking does not show meaningfully stronger uplift in its top decile.



<!-- CELL 14 — MARKDOWN -->
## 7. Combine all rankings into one DataFrame

In [7]:
# ---------------------------------------------------------------------------
# CELL 15 — CODE: Build one combined DataFrame with every model's uplift
# score plus the Phase 3 baseline's predicted probability, joined on
# row_index, so Phase 5 can compare all four rankings on the same customers.
# ---------------------------------------------------------------------------
combined = pd.DataFrame({
    "row_index": X_test.index,
    "actual_treatment": treatment_test.values,
    "actual_visit": y_test["visit"].values,
    "actual_conversion": y_test["conversion"].values,
    "uplift_two_model": uplift_two_model,
    "uplift_class_transform": uplift_class_transform,
    "uplift_causal_forest": uplift_causal_forest,
})

baseline_ranking = pd.read_csv(os.path.join(processed_dir, "baseline_ranking.csv"))
baseline_lookup = baseline_ranking[["row_index", "predicted_prob"]].rename(
    columns={"predicted_prob": "baseline_predicted_prob"}
)

combined = combined.merge(baseline_lookup, on="row_index", how="left")

assert combined["baseline_predicted_prob"].isna().sum() == 0, \
    "Some rows failed to join with the Phase 3 baseline ranking -- row_index mismatch."

print(f"Combined DataFrame: {combined.shape[0]:,} rows x {combined.shape[1]} columns")
print(f"Columns: {combined.columns.tolist()}")
combined.head()


Combined DataFrame: 12,800 rows x 8 columns
Columns: ['row_index', 'actual_treatment', 'actual_visit', 'actual_conversion', 'uplift_two_model', 'uplift_class_transform', 'uplift_causal_forest', 'baseline_predicted_prob']


,row_index,actual_treatment,actual_visit,actual_conversion,uplift_two_model,uplift_class_transform,uplift_causal_forest,baseline_predicted_prob
0,0,0,0,0,0.199408,-0.199842,0.070376,0.413840
1,1,1,0,0,0.085988,-0.123124,0.062138,0.585578
2,2,0,1,1,-0.141157,-0.218892,0.042147,0.507057
3,3,1,0,0,0.213011,-0.276558,0.068648,0.208846
4,4,0,0,0,0.412646,-0.070728,0.149769,0.497651


<!-- CELL 16 — MARKDOWN -->
## 8. Save outputs

In [8]:
# ---------------------------------------------------------------------------
# CELL 17 — CODE: Save the combined rankings CSV and the causal forest
# model object, so Phase 5 can load both directly.
# ---------------------------------------------------------------------------
combined_path = os.path.join(processed_dir, "uplift_scores_combined.csv")
combined.to_csv(combined_path, index=False)
print(f"Saved combined rankings -> {combined_path}  ({os.path.getsize(combined_path)/1024:.1f} KB)")

causal_forest_path = os.path.join(processed_dir, "causal_forest_model.pkl")
joblib.dump(causal_forest_model, causal_forest_path)
print(f"Saved causal forest model -> {causal_forest_path}  ({os.path.getsize(causal_forest_path)/1024:.1f} KB)")


Saved combined rankings -> ..\data\processed\uplift_scores_combined.csv  (1152.8 KB)
Saved causal forest model -> ..\data\processed\causal_forest_model.pkl  (87372.4 KB)


<!-- CELL 18 — MARKDOWN -->
## 9. Phase 4 summary

In [9]:
# ---------------------------------------------------------------------------
# CELL 19 — CODE: Identify the strongest model from Section 6's signal
# verification and pull together the closing summary for Phase 4.
# ---------------------------------------------------------------------------
best_model_name = max(verification_results, key=lambda k: verification_results[k]["gap"])
best_gap = verification_results[best_model_name]["gap"]

results_table = pd.DataFrame(verification_results).T
results_table.index.name = "model"
results_table = results_table[["top_decile_uplift", "bottom_90pct_uplift", "gap", "passed"]]
display(results_table)

results_rows_md = "\n".join(
    f"| {name} | {res['top_decile_uplift']*100:+.2f} pp | {res['bottom_90pct_uplift']*100:+.2f} pp | "
    f"{res['gap']*100:+.2f} pp | {'Yes' if res['passed'] else 'No'} |"
    for name, res in verification_results.items()
)

summary_md = f"""
### Phase 4 Summary — Uplift Models

- **Causal forest library used:** `{CAUSAL_LIBRARY}` (see Section 2 for why).
- **Three uplift models trained:** Two-Model Approach, Class Transformation
  ("revert label," Gutierrez & Gerardy 2017), and Causal Forest
  (Athey, Tibshirani & Wager 2019) -- all on `visit` as the outcome, consistent with
  Phase 3's corrected target.
- **Signal verification (top decile vs. bottom 90% real uplift):**

| Model | Top-decile actual uplift | Bottom-90% actual uplift | Gap | Passed |
|---|---|---|---|---|
{results_rows_md}

- **Strongest model by this check: `{best_model_name}`**, with a top-decile-vs-rest actual
  uplift gap of **{best_gap*100:+.2f} pp** -- meaning its highest-ranked 10% of test
  customers showed a real (not just predicted) treated-vs-control visit-rate gap that much
  larger than everyone else, which is exactly the property a useful targeting model needs.
- **Outputs saved:** `data/processed/uplift_scores_combined.csv` ({len(combined):,} test
  customers, all 3 uplift scores + the Phase 3 baseline's predicted probability, joined on
  `row_index`) and `data/processed/causal_forest_model.pkl` (the fitted Causal Forest object).

**Next steps (Phase 5):** this notebook's top-decile-vs-rest check is a quick, intuitive
sanity check, but it only looks at one point (the top 10%) and doesn't account for how the
gap evolves as more customers are targeted. **Phase 5 will formally evaluate all four
rankings (baseline + 3 uplift models) with Qini coefficients and Qini curves**, which
summarize a model's uplift-ranking quality across the *entire* targeting range, not just
the top decile.
"""

display(Markdown(summary_md))


,top_decile_uplift,bottom_90pct_uplift,gap,passed
model,,,,
Two-Model Approach,0.085846,0.052275,0.033571,True
Class Transformation,0.071191,0.053489,0.017702,True
Causal Forest,0.046998,0.056639,-0.009641,False



### Phase 4 Summary — Uplift Models

- **Causal forest library used:** `econml` (see Section 2 for why).
- **Three uplift models trained:** Two-Model Approach, Class Transformation
  ("revert label," Gutierrez & Gerardy 2017), and Causal Forest
  (Athey, Tibshirani & Wager 2019) -- all on `visit` as the outcome, consistent with
  Phase 3's corrected target.
- **Signal verification (top decile vs. bottom 90% real uplift):**

| Model | Top-decile actual uplift | Bottom-90% actual uplift | Gap | Passed |
|---|---|---|---|---|
| Two-Model Approach | +8.58 pp | +5.23 pp | +3.36 pp | Yes |
| Class Transformation | +7.12 pp | +5.35 pp | +1.77 pp | Yes |
| Causal Forest | +4.70 pp | +5.66 pp | -0.96 pp | No |

- **Strongest model by this check: `Two-Model Approach`**, with a top-decile-vs-rest actual
  uplift gap of **+3.36 pp** -- meaning its highest-ranked 10% of test
  customers showed a real (not just predicted) treated-vs-control visit-rate gap that much
  larger than everyone else, which is exactly the property a useful targeting model needs.
- **Outputs saved:** `data/processed/uplift_scores_combined.csv` (12,800 test
  customers, all 3 uplift scores + the Phase 3 baseline's predicted probability, joined on
  `row_index`) and `data/processed/causal_forest_model.pkl` (the fitted Causal Forest object).

**Next steps (Phase 5):** this notebook's top-decile-vs-rest check is a quick, intuitive
sanity check, but it only looks at one point (the top 10%) and doesn't account for how the
gap evolves as more customers are targeted. **Phase 5 will formally evaluate all four
rankings (baseline + 3 uplift models) with Qini coefficients and Qini curves**, which
summarize a model's uplift-ranking quality across the *entire* targeting range, not just
the top decile.
